# 4 — Grid-Tie VSI: SRF-PLL + dq Current Control

> **Goal.** Connect the VSI to a stiff AC grid. Two new things:
> 1) a **PLL** locks onto the grid angle (we no longer generate
> $\theta$ internally), and 2) the controlled quantity changes from
> *voltage* to *current* — we inject a target $i_d$ (active current)
> and $i_q$ (reactive current) that map directly to **P** and **Q**.

**Prerequisites**

- All previous notebooks in this project, especially `01` (transforms)
- Boost PFC CCM notebook (`projects/converters/boost_pfc/03_boost_pfc_ccm.ipynb`)
  — the inner current loop architecture transfers verbatim.

**What you'll be able to do at the end**

1. Describe the **synchronous reference frame PLL** (SRF-PLL) — the
   standard grid-synchronization technique.
2. Derive the dq current-loop plant for a grid-connected inverter
   (a first-order $L_{grid}$ — no LC filter on this side, just the
   coupling inductor).
3. Design the **two PI compensators** (current loop in dq) and the
   **PLL PI** independently.
4. Explain **cross-coupling decoupling**: feed-forward the $\omega
   L_{grid} i_d$ and $\omega L_{grid} i_q$ terms to make the dq
   axes truly independent.
5. Run the full switched closed-loop sim with **P-Q injection** and
   verify that $i_d^{ref}$ produces the expected active power
   transfer.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from vsi_3phase_model import (
    VSI3PhaseParams,
    simulate_closed_loop_gridtie,
    clarke_transform, park_transform,
    operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

p = VSI3PhaseParams()
print(operating_point_report(p, mode="gridtie"))


## 1. Grid-tie architecture

A grid-tie inverter:

- **Source**: DC bus (typically from a PFC or PV array).
- **Sink**: stiff AC grid via a coupling inductor $L_{grid}$.
- **No LC filter** on the grid side (the grid is the cap) — just
  $L_{grid}$ to absorb switching ripple.
- **Output current** is what we control (not voltage). The grid sets
  the voltage.

```
   DC bus                              v_grid
     │                                    │
   ──┴── ── inverter ── L_grid ── ────────┤
                                          │
                                       (stiff)
```

Within the controller:

```
   v_grid_abc → Clarke → αβ → Park (θ_pll) → dq → drive v_qg → 0 (PLL)
                                                     │
                                                     ↓
                                                  θ_pll (angle)
                                                     │
   i_grid_abc → Clarke → αβ → Park (θ_pll) → dq ─┐
                                                  │
                                                  ↓
   i_d_ref ─┐                                   PI in dq
   i_q_ref ─┴── err ── PI ──→ v_d_cmd, v_q_cmd ─┘
                                  │
                                  ↓
                              inv. Park → αβ → inv. Clarke → SVPWM → gates
```

The PLL produces $\theta_{pll}$ that tracks the grid voltage angle;
the current loop runs in this frame.


## 2. SRF-PLL (synchronous reference frame PLL)

How it works:

1. Sample $v_{ga,b,c}$ and Clarke-transform to αβ.
2. Park-transform using the **current PLL angle estimate** $\theta_{pll}$
   to get $v_{dg}$ and $v_{qg}$.
3. If $\theta_{pll}$ lags the true angle, $v_{qg}$ becomes positive.
   If it leads, $v_{qg}$ becomes negative. So **drive $v_{qg} → 0$**
   to lock the phase.
4. PI compensator on $v_{qg}$ produces a frequency correction
   $\Delta\omega$. Add to the nominal grid frequency to get the
   instantaneous $\omega_{pll}$. Integrate to get $\theta_{pll}$.

Plant for the PLL: a pure integrator (from $\omega_{pll}$ to
$\theta_{pll}$). PI design is straightforward — fast loop (~50 Hz
crossover or so) for clean tracking.


In [ ]:
# Design the PLL PI: input = -v_qg (with sign so positive err raises ω)
# Plant: 1/s integrator (theta) + the trig in Park is at unity DC gain near lock
# Target f_c = 50 Hz, zero ≈ 5 Hz
T_s = p.T_sw

# Effective plant gain at lock: ∂v_qg/∂Δω ≈ V_grid_pk · (Δθ ≈ Δω·t_settle)
V_grid_pk = p.V_grid_LL_rms * np.sqrt(2.0) / np.sqrt(3.0)
# Plant from omega_corr (rad/s) to v_qg (V) at lock ≈ V_grid_pk · (1/s)
# So G_pll(s) = V_grid_pk / s
# Compensator: K · (1 + s·tau_z) / s, target crossover 50 Hz
omega_c_pll = 2*np.pi*50.0
tau_z_pll = 1.0/(2*np.pi*5.0)
# At omega_c: |compensator| = K · sqrt(1+(omega_c·tau)²)/omega_c
# Loop gain = V_grid_pk/omega_c · compensator
# Want = 1: K = omega_c² / (V_grid_pk · sqrt(1+(omega_c·tau)²))
K_pll = omega_c_pll**2 / (V_grid_pk * np.sqrt(1+(omega_c_pll*tau_z_pll)**2))
print(f"PLL PI: K = {K_pll:.4g}, tau_z = {tau_z_pll*1e3:.2f} ms (zero at 5 Hz)")
Gc_pll = signal.TransferFunction([K_pll*tau_z_pll, K_pll], [1.0, 0.0])
bp_raw, ap_raw, _ = signal.cont2discrete((Gc_pll.num, Gc_pll.den), dt=T_s, method='bilinear')
b_pll = np.asarray(bp_raw).flatten()/ap_raw[0]
a_pll = np.asarray(ap_raw)/ap_raw[0]
print(f"  Discrete: b = {b_pll}, a = {a_pll}")


## 3. Current-loop design

Plant from $v_{d,cmd}$ to $i_d$, within the current-loop bandwidth
(grid is stiff, $v_d$ ≈ $V_{d,grid}$ const):

$$
L_{grid} \frac{d i_d}{dt} = v_{d,cmd} - v_{d,grid} - \omega L_{grid} i_q
$$

The $-v_{d,grid}$ and $-\omega L_{grid} i_q$ are **disturbances**
that we can feed-forward. After feed-forward, the current loop's
effective plant is a clean integrator:

$$
G_{id}(s) = \frac{1}{s L_{grid}}
$$

Same as the boost PFC's current loop! Use the same recipe: PI with
zero placed 3× below crossover at $f_c = 1$ kHz.


In [ ]:
# Current-loop PI design (after feed-forward)
f_c_i = 1000.0  # 1 kHz current-loop bandwidth
omega_c_i = 2*np.pi*f_c_i
tau_z_i = 3.0 / omega_c_i  # zero at f_c/3

# Plant at omega_c: 1/(omega_c · L_grid)
plant_mag_i = 1.0 / (omega_c_i * p.L_grid)
K_i = omega_c_i / (np.sqrt(1+(omega_c_i*tau_z_i)**2) * plant_mag_i)
print(f"Current PI: K = {K_i:.4g}, tau_z = {tau_z_i*1e6:.2f} µs "
      f"(zero at {1/(2*np.pi*tau_z_i)/1000:.2f} kHz)")
Gc_i = signal.TransferFunction([K_i*tau_z_i, K_i], [1.0, 0.0])
bi_raw, ai_raw, _ = signal.cont2discrete((Gc_i.num, Gc_i.den), dt=T_s, method='bilinear')
bi = np.asarray(bi_raw).flatten()/ai_raw[0]
ai = np.asarray(ai_raw)/ai_raw[0]


## 4. P/Q injection via dq current references

In the grid frame with PLL locked so $v_{qg} = 0$ (i.e. $v_{d,grid}
= V_{d,pk}$):

$$
P = \frac{3}{2} V_{d,grid} \cdot i_d, \qquad
Q = -\frac{3}{2} V_{d,grid} \cdot i_q
$$

(The $3/2$ comes from the amplitude-invariant Clarke convention.)

To inject 500 W of active power into the grid at the nominal grid
voltage:
$$
I_{d,ref} = \frac{2 P_{ref}}{3 V_{d,grid}}
$$


In [ ]:
P_ref = 500.0  # 500 W active injection
Q_ref = 0.0    # zero reactive (unity PF injection)
I_d_ref = 2 * P_ref / (3 * V_grid_pk)
I_q_ref = -2 * Q_ref / (3 * V_grid_pk)
print(f"For P = {P_ref:.0f} W, Q = {Q_ref:.0f} VAR:")
print(f"  I_d_ref = {I_d_ref:.4f} A   (peak active current)")
print(f"  I_q_ref = {I_q_ref:.4f} A   (peak reactive current)")
print(f"  I_pk    = {np.sqrt(I_d_ref**2 + I_q_ref**2):.4f} A")


## 5. Switched closed-loop simulation

Run the full grid-tie simulation: PLL + current loop + multiplier-free
power calculation. The current $i_a$ should be a clean sinusoid in
phase with $v_{ga}$ (unity PF injection).


In [ ]:
sim = simulate_closed_loop_gridtie(p, bi, ai, b_pll, a_pll,
                                    I_d_ref=I_d_ref, I_q_ref=I_q_ref,
                                    n_cycles=10, samples_per_period=80,
                                    use_svpwm=True)
print(f"Simulated {len(sim['t'])} samples over {sim['t'][-1]*1000:.1f} ms")


In [ ]:
mask = sim['t'] > 3.0/p.f_grid

fig, axs = plt.subplots(4, 1, figsize=(12, 11), sharex=True)

axs[0].plot(sim['t']*1000, sim['v_ga'], "C0", linewidth=1.0, label="$v_{g,a}$")
ax_i = axs[0].twinx()
ax_i.plot(sim['t']*1000, sim['i_a'], "C3", linewidth=1.0, label="$i_a$")
axs[0].set_ylabel("Grid voltage [V]"); ax_i.set_ylabel("Grid current [A]", color="C3")
axs[0].set_title(f"Grid-tie VSI: P_ref = {P_ref:.0f} W (unity-PF injection)")
axs[0].legend(loc="upper left"); ax_i.legend(loc="upper right")

axs[1].plot(sim['t']*1000, sim['theta_pll'], "C2", linewidth=1.0, label="$\\theta_{pll}$")
axs[1].plot(sim['t']*1000, sim['theta_true'], "C5--", linewidth=1.0, label="$\\theta_{true}$")
axs[1].set_ylabel("Angle [rad]"); axs[1].legend(loc="lower right")

axs[2].plot(sim['t']*1000, sim['i_d'], "C2", linewidth=1.2, label="$i_d$")
axs[2].axhline(I_d_ref, color="C2", linestyle=":", alpha=0.5,
               label=f"$i_d^{{ref}}$ = {I_d_ref:.3f} A")
axs[2].plot(sim['t']*1000, sim['i_q'], "C1", linewidth=1.2, label="$i_q$")
axs[2].axhline(I_q_ref, color="C1", linestyle=":", alpha=0.5,
               label=f"$i_q^{{ref}}$ = {I_q_ref:.3f} A")
axs[2].set_ylabel("Current dq [A]"); axs[2].legend(loc="upper right", fontsize=8)

axs[3].plot(sim['t']*1000, sim['omega_pll']/(2*np.pi), "C4", linewidth=1.0)
axs[3].axhline(p.f_grid, color="k", linestyle=":", alpha=0.4,
               label=f"true f_grid = {p.f_grid:.1f} Hz")
axs[3].set_ylabel("PLL freq [Hz]"); axs[3].set_xlabel("Time [ms]")
axs[3].legend()

plt.tight_layout(); plt.show()


In [ ]:
# Verify P/Q injection
i_d_ss = sim['i_d'][mask].mean()
i_q_ss = sim['i_q'][mask].mean()
P_actual = 1.5 * V_grid_pk * i_d_ss
Q_actual = -1.5 * V_grid_pk * i_q_ss
print("Steady-state P/Q injection:")
print(f"  i_d steady = {i_d_ss:.4f} A  (target {I_d_ref:.4f} A)")
print(f"  i_q steady = {i_q_ss:.4f} A  (target {I_q_ref:.4f} A)")
print(f"  P actual   = {P_actual:.2f} W   (target {P_ref:.2f} W, "
      f"error {(P_actual-P_ref)/P_ref*100:+.2f}%)")
print(f"  Q actual   = {Q_actual:.2f} VAR (target {Q_ref:.2f} VAR)")
print()

# Verify PLL locked
theta_err = sim['theta_pll'] - sim['theta_true']
# Unwrap to range [-pi, pi]
theta_err = np.mod(theta_err + np.pi, 2*np.pi) - np.pi
theta_err_ss = theta_err[mask].mean()
print(f"PLL lock check:")
print(f"  theta_pll - theta_true (steady state) = {np.rad2deg(theta_err_ss):.3f}°")
print(f"  omega_pll (steady state) = {sim['omega_pll'][mask].mean()/(2*np.pi):.4f} Hz "
      f"(target {p.f_grid:.4f} Hz)")

if abs(P_actual - P_ref) < 50 and abs(theta_err_ss) < np.deg2rad(2):
    print()
    print("✅  Grid-tie VSI PROVEN: PLL locked, P injected within ±10% of target.")


## 6. Summary

Grid-tie VSI combines two control loops with very different time
scales:

- **Inner current loop** (1 kHz crossover): forces $i_d$ and $i_q$
  to track references in the dq frame; plant is an integrator
  ($1/(sL_{grid})$) after grid-voltage feed-forward. Simple PI.
- **SRF-PLL** (50 Hz crossover): tracks grid angle by driving
  $v_{qg} → 0$. Simple PI on a virtual integrator plant.

Together they let you inject **arbitrary $P$ and $Q$** with just
two reference numbers ($i_d^{ref}, i_q^{ref}$). The math is
identical whether you're injecting from a PV array, a battery, or
absorbing into an EV charger.

**What we didn't cover** (production additions):

- **Anti-islanding** detection (required by grid codes)
- **LCL filter** instead of L (lower switching ripple, but adds a
  resonance to damp)
- **Negative-sequence rejection** for unbalanced grids
- **Weak-grid PLL stabilization** (the PLL itself can oscillate
  when the grid is "soft" — small $S_k$)
- **Fault ride-through** logic (stay connected during voltage sags)
- **MPPT integration** with a PV upstream stage

**Suggested exercises**

1. Step $I_d^{ref}$ from 0 to nominal at $t = 50$ ms. Plot the
   current-loop step response. Settling time?
2. Change $I_q^{ref}$ to inject 200 VAR of reactive power. Does
   the active-current loop see any cross-coupling?
3. Detune the PLL gain (× 0.1). Does it still lock? How long?
4. Add a 5 Hz frequency jitter to the grid (`omega_grid_true ·=
   1 + 0.01 sin(2π·5·t)`). Does the PLL track? Plot
   $\theta_{pll} - \theta_{true}$.
5. Implement cross-coupling decoupling in the controller — subtract
   $\omega L_{grid} i_q$ from $v_{d,cmd}$ and add $\omega L_{grid} i_d$
   to $v_{q,cmd}$. (It's already in `simulate_closed_loop_gridtie`.)
   Compare with and without.

## Library complete (for now)

| Project | I/O | Switches | Loops |
|---|---|---|---|
| 6 DC-DC | DC → DC | 1 or 2 | 1 voltage |
| boost PFC (2 modes) | AC → DC | 1 | 1 or 2 |
| **VSI 3-phase stand-alone** | **DC → AC** | **6** | **2 (d + q)** |
| **VSI 3-phase grid-tie** | **DC ↔ AC** | **6** | **3 (PLL + d + q current)** |

You've now seen the full power-electronics control hierarchy, from
single-switch buck up to multi-loop grid-synchronized inverters. The
reference-frame transforms are the most powerful idea — they let
3-phase AC control reuse single-axis DC control intuition.
